In this notebook I will create a training table equivalent to ml_model_run_details

# Define Library

In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid

import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\application_default_credentials.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


### Function

#### expand_calc_features

In [2]:
import pandas as pd
import json

def expand_calc_features(df):
    """
    Expand the calcFeatures JSON column into separate columns and return the complete DataFrame.

    Parameters:
    df (pd.DataFrame): Input DataFrame with calcFeatures column containing JSON data

    Returns:
    pd.DataFrame: Expanded DataFrame with all original columns plus JSON features as separate columns
    """

    # Make a copy to avoid modifying the original DataFrame
    df_expanded = df.copy()

    # Parse the calcFeatures JSON column
    calc_features_list = []

    for idx, calc_features_str in enumerate(df['calcFeatures']):
        try:
            # Parse the JSON string
            features_dict = json.loads(calc_features_str.replace("'", '"'))  # Replace single quotes with double quotes for valid JSON
            calc_features_list.append(features_dict)
        except (json.JSONDecodeError, AttributeError) as e:
            # If parsing fails, create an empty dict and print warning
            print(f"Warning: Could not parse calcFeatures at index {idx}: {e}")
            calc_features_list.append({})

    # Create DataFrame from the parsed JSON data
    calc_features_df = pd.DataFrame(calc_features_list)

    # Add prefix to JSON-derived columns to avoid conflicts
    calc_features_df = calc_features_df.add_prefix('calc_')

    # Reset index to ensure proper alignment
    df_expanded = df_expanded.reset_index(drop=True)
    calc_features_df = calc_features_df.reset_index(drop=True)

    # Combine original DataFrame with expanded calcFeatures
    result_df = pd.concat([df_expanded, calc_features_df], axis=1)

    return result_df


#### expand_calc_features_robust

In [3]:
import pandas as pd
import json

def expand_calc_features_robust(df):
    """
    Expand the calcFeatures JSON column into separate columns with better error handling.

    Parameters:
    df (pd.DataFrame): Input DataFrame with calcFeatures column containing JSON data

    Returns:
    pd.DataFrame: Expanded DataFrame with all original columns plus JSON features as separate columns
    """

    # Make a copy to avoid modifying the original DataFrame
    df_expanded = df.copy()

    # Parse the calcFeatures JSON column
    calc_features_data = []

    for idx, row in df.iterrows():
        calc_features_str = row['calcFeatures']

        if pd.isna(calc_features_str) or calc_features_str == '':
            calc_features_data.append({})
            continue

        try:
            # Clean the string and parse JSON
            cleaned_str = calc_features_str.replace("'", '"').replace('None', 'null').replace('True', 'true').replace('False', 'false')
            features_dict = json.loads(cleaned_str)
            calc_features_data.append(features_dict)
        except Exception as e:
            print(f"Warning: Could not parse calcFeatures at index {idx}: {e}")
            print(f"Problematic string: {calc_features_str[:100]}...")  # Print first 100 chars
            calc_features_data.append({})

    # Create DataFrame from the parsed JSON data
    calc_features_df = pd.DataFrame(calc_features_data)

    # Add prefix to JSON-derived columns to avoid conflicts with existing columns
    calc_features_df = calc_features_df.add_prefix('feat_')

    # Combine DataFrames
    result_df = pd.concat([df_expanded, calc_features_df], axis=1)

    print(f"Original DataFrame shape: {df.shape}")
    print(f"Expanded DataFrame shape: {result_df.shape}")
    print(f"Added {len(calc_features_df.columns)} new columns from calcFeatures")

    return result_df

# Table Name

In [16]:
table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"

# Transform data v1.1

In [7]:
import pandas as pd
import json
import uuid
from datetime import datetime
from typing import List

def transform_data_v1_1(
    d1: pd.DataFrame, 
    feature_column: List[str], 
    a: str = 'demo_score', 
    modelDisplayName: str = 'Cash_beta_trench1_Demo_backscore', 
    tc: str = "", 
    subscription_name: str = 'sil_march 25 models'
) -> pd.DataFrame:
    """
    Transforms input data into a structured format suitable for model scoring output.

    Parameters:
    - d1 (pd.DataFrame): Input DataFrame containing raw data.
    - feature_column (List[str]): List of column names to include in the 'calcFeature' JSON.
    - a (str): Column name containing the prediction score. Default is 'demo_score'.
    - modelDisplayName (str): Name of the model used for scoring.
    - tc (str): Trench category (optional).
    - do (str): Device operating system. Default is 'android'.
    - subscription_name (str): Name of the subscription or model group.

    Returns:
    - pd.DataFrame: Transformed DataFrame with structured output.
    """

    # Make a copy of the input DataFrame to avoid modifying the original
    df = d1.copy()
    
    # Initialize an empty list to store transformed rows
    output_data = []
    
    # Iterate over each row in the DataFrame
    for _, row in df.iterrows():
        # Initialize dictionary to hold feature values
        calc_feature = {}
        
        # Loop through each feature column and extract its value from the row
        for col in feature_column:
            if col in row and pd.notna(row[col]):
                # Convert datetime values to ISO format strings
                if isinstance(row[col], pd.Timestamp):
                    calc_feature[col] = row[col].isoformat()
                else:
                    calc_feature[col] = row[col]
        
        # Get the current timestamp for start_time, end_time, and publish_time
        current_time = datetime.now().isoformat()
        
        # Construct the output row dictionary with required fields
        output_row = {
            "customerId": row['customer_id'],  # Unique customer identifier
            "digitalLoanAccountId": row['digitalLoanAccountId'],  # Loan account ID
            "crifApplicationId": str(uuid.uuid4()),  # Random UUID for application ID
            "prediction": row.get(a, 0),  # Prediction score from specified column
            "start_time": current_time,  # Timestamp when processing starts
            "end_time": current_time,    # Timestamp when processing ends
            "modelDisplayName": modelDisplayName,  # Name of the model used
            "modelVersionId": "v1.1",  # Static model version
            "calcFeature": json.dumps(calc_feature, default=str),  # Features as JSON string
            "subscription_name": subscription_name,  # Subscription name
            "message_id": str(uuid.uuid4()),  # Random UUID for message ID
            "publish_time": current_time,  # Timestamp when message is published
            "attributes": "{}",  # Placeholder for additional attributes
            "trenchCategory": tc,  # Optional trench category
            "deviceOs": row['osType'],
            "Data_selection": row['Data_selection'],  # Data selection
            "Application_date": row['application_date'],
        }
        
        # Append the transformed row to the output list
        output_data.append(output_row)
    
    # Convert the list of dictionaries to a DataFrame
    output_df = pd.DataFrame(output_data)
    
    # Return the transformed DataFrame
    return output_df


# Version 1.1

## Cash

# apps_score_cash

## Trench 1

risk_mart.quick_applied_aug2024_nov2025_app_scored_extended only had cb_score and dl_score and was lacking other features used for cb_score, Oleh gave another table select * from  risk_mart.quick_applied_aug2024_nov2025_app_scored_extended_v2;

In [ ]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customerId,
      r.digitalLoanAccountId,
      r.application_submission_date,
      CASE
        WHEN r.trenchCategory LIKE 'T1-A' THEN 'Trench 1'
        WHEN r.trenchCategory LIKE 'T2-A' THEN 'Trench 2'
        WHEN r.trenchCategory LIKE 'T3-A' THEN 'Trench 3'
        END trenchCategory,
        r.apps_score,
        r.cb_score ml_score,
        r.dl_score,
        r.app_cnt_absence_tag_90d,	
        r.app_cnt_rated_for_18plus_ever,	
        r.app_last_payday_install_to_apply_days,	
        r.app_cnt_lifestyle_ever,
        r.app_median_time_bw_installed_mins_ever,
        r.app_cnt_payday_ever,
        r.app_cnt_food_and_drink_ever,
        r.app_cnt_gaming_180d,
        r.app_vel_payday_30_over_365,
        r.app_first_app_cat,
        lower(r.deviceOs) osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
      CASE
        WHEN dataset LIKE 'Dev_Train' THEN 'Dev_Train'
        WHEN dataset LIKE 'Dev_Test' THEN 'Dev_Test'
        WHEN dataset LIKE 'OOS' THEN 'Dev_Test'
        END Data_selection
    FROM risk_mart.quick_applied_aug2024_nov2025_app_scored_extended_v2 r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
  )
SELECT
  customerId customer_id,
  digitalLoanAccountId,
  application_submission_date,
  trenchCategory,
  apps_score,
  ml_score,
  dl_score,
  app_cnt_absence_tag_90d,	
  app_cnt_rated_for_18plus_ever,	
  app_last_payday_install_to_apply_days,	
  app_cnt_lifestyle_ever,
  app_median_time_bw_installed_mins_ever,
  app_cnt_payday_ever,
  app_cnt_food_and_drink_ever,
  app_cnt_gaming_180d,
  app_vel_payday_30_over_365,
  app_first_app_cat,
  osType,
  application_date,
  Data_selection
FROM base
WHERE
  trenchCategory = 'Trench 1'
  AND apps_score IS NOT NULL;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.columns

In [ ]:
feature_column = ['apps_score', 'ml_score', 'dl_score',
       'app_cnt_absence_tag_90d', 'app_cnt_rated_for_18plus_ever',
       'app_last_payday_install_to_apply_days', 'app_cnt_lifestyle_ever',
       'app_median_time_bw_installed_mins_ever', 'app_cnt_payday_ever',
       'app_cnt_food_and_drink_ever', 'app_cnt_gaming_180d',
       'app_vel_payday_30_over_365', 'app_first_app_cat',]

dfd = transform_data_v1_1(data, feature_column, a='apps_score', modelDisplayName='apps_score_cash', tc='Trench 1', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
dfd.head()

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_application"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

## Trench 2

In [ ]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customerId,
      r.digitalLoanAccountId,
      r.application_submission_date,
      CASE
        WHEN r.trenchCategory LIKE 'T1-A' THEN 'Trench 1'
        WHEN r.trenchCategory LIKE 'T2-A' THEN 'Trench 2'
        WHEN r.trenchCategory LIKE 'T3-A' THEN 'Trench 3'
        END trenchCategory,
        r.apps_score,
        r.cb_score ml_score,
        r.dl_score,
        r.app_cnt_absence_tag_90d,	
        r.app_cnt_rated_for_18plus_ever,	
        r.app_last_payday_install_to_apply_days,	
        r.app_cnt_lifestyle_ever,
        r.app_median_time_bw_installed_mins_ever,
        r.app_cnt_payday_ever,
        r.app_cnt_food_and_drink_ever,
        r.app_cnt_gaming_180d,
        r.app_vel_payday_30_over_365,
        r.app_first_app_cat,
    lower(r.deviceOs)  osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
      CASE
        WHEN dataset LIKE 'Dev_Train' THEN 'Dev_Train'
        WHEN dataset LIKE 'Dev_Test' THEN 'Dev_Test'
        WHEN dataset LIKE 'OOS' THEN 'Dev_Test'
        END Data_selection
    FROM risk_mart.quick_applied_aug2024_nov2025_app_scored_extended_v2 r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
  )
SELECT
  customerId customer_id,
  digitalLoanAccountId,
  application_submission_date,
  trenchCategory,
  apps_score,
  ml_score,
  dl_score,
  app_cnt_absence_tag_90d,	
  app_cnt_rated_for_18plus_ever,	
  app_last_payday_install_to_apply_days,	
  app_cnt_lifestyle_ever,
  app_median_time_bw_installed_mins_ever,
  app_cnt_payday_ever,
  app_cnt_food_and_drink_ever,
  app_cnt_gaming_180d,
  app_vel_payday_30_over_365,
  app_first_app_cat,
  osType,
  application_date,
  Data_selection
FROM base
WHERE
  trenchCategory = 'Trench 2'
  AND apps_score IS NOT NULL;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
feature_column = ['apps_score', 'ml_score', 'dl_score',
       'app_cnt_absence_tag_90d', 'app_cnt_rated_for_18plus_ever',
       'app_last_payday_install_to_apply_days', 'app_cnt_lifestyle_ever',
       'app_median_time_bw_installed_mins_ever', 'app_cnt_payday_ever',
       'app_cnt_food_and_drink_ever', 'app_cnt_gaming_180d',
       'app_vel_payday_30_over_365', 'app_first_app_cat',]

dfd = transform_data_v1_1(data, feature_column, a='apps_score', modelDisplayName='apps_score_cash', tc='Trench 2', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
dfd.head()

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

# Beta_stack_Model_cash

## Trench 1

In [ ]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customer_id,
      r.digitalLoanAccountId,
      r.ln_appln_submit_datetime,
      r.trench_category,
      r.demo_score,
      r.apps_score,
      r.credo_score,
      r.stack_score,
      lower(r.ln_os_type) osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
      -- CASE
      --   WHEN dev_split LIKE 'Dev_Train' THEN 'Dev_Train'
      --   WHEN dev_split LIKE 'Dev_Test' THEN 'Dev_Test'
      --   ELSE dev_split
      --   END Data_selection
      case when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) < '2025-06-01' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2025-06-01' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) <= '2025-02-28' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) > '2025-02-28' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test' 
           end Data_selection
      from 
      `worktable_data_analysis.cash_beta_trench1_applied_loans_backscored_20241001_20251130`       r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
    where stack_score is not null
      )
      ,
b1 as 
(SELECT 
customer_id,
digitalLoanAccountId,
ln_appln_submit_datetime,
trench_category,
demo_score,
apps_score,
credo_score,
stack_score Beta_cash_stack_score,
osType,
application_date,
Data_selection,
FROM base
WHERE Data_selection IS NOT NULL
)
select * from b1 

"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'Beta_cash_stack_score']

dfd = transform_data_v1_1(data, feature_column, a='Beta_cash_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
dfd.head()

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

## Trench 2

In [ ]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customer_id,
      r.digitalLoanAccountId,
      r.ln_appln_submit_datetime,
      r.trench_category,
      r.demo_score,
      r.apps_score,
      r.credo_score,
      r.stack_score,
      lower(r.ln_os_type) osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
      -- CASE
      --   WHEN dev_split LIKE 'Dev_Train' THEN 'Dev_Train'
      --   WHEN dev_split LIKE 'Dev_Test' THEN 'Dev_Test'
      --   ELSE dev_split
      --   END Data_selection
      case when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) < '2025-06-01' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2025-06-01' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) <= '2025-02-28' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) > '2025-02-28' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test' 
           end Data_selection
      from 
      `worktable_data_analysis.cash_beta_trench2_applied_loans_backscored_20241001_20251130` r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
    where stack_score is not null
      )
      ,
b1 as 
(SELECT 
customer_id,
digitalLoanAccountId,
ln_appln_submit_datetime,
trench_category,
demo_score,
apps_score,
credo_score,
stack_score Beta_cash_stack_score,
osType,
application_date,
Data_selection,
FROM base
WHERE Data_selection IS NOT NULL
)
select * from b1 
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
data.head()

In [ ]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'Beta_cash_stack_score', 'stack_score']

dfd = transform_data_v1_1(data, feature_column, a='Beta_cash_stack_score', modelDisplayName='Beta-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

In [ ]:
dfd.head()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

# Alpha Cash Stack Model

## Trench 1

In [ ]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customer_id,
      r.digitalLoanAccountId,
      r.ln_appln_submit_datetime,
      r.trench_category,
      r.demo_score,
      r.apps_score,
      r.credo_score,
      r.stack_score,
      r.cic_score,
      lower(r.ln_os_type)  osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
       case when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) < '2025-06-01' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2025-06-01' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) <= '2025-02-28' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) > '2025-02-28' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test' 
           end Data_selection
      -- CASE
      --   WHEN dev_split LIKE 'Dev_Train' THEN 'Dev_Train'
      --   WHEN dev_split LIKE 'Dev_Test' THEN 'Dev_Test'
      --   ELSE dev_split
      --   END Data_selection
    FROM
      `worktable_data_analysis.cash_alpha_trench1_applied_loans_backscored_20241001_20251130`
        r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
    WHERE stack_score is not null
  ),
  b1 as 
(SELECT 
customer_id,
digitalLoanAccountId,
ln_appln_submit_datetime,
trench_category,
demo_score,
apps_score,
credo_score,
stack_score,
cic_score,
osType,
application_date,
Data_selection,
FROM base
WHERE Data_selection IS NOT NULL
)
select * from b1;

"""
data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")


In [ ]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'cic_score', 'stack_score']

dfd = transform_data_v1_1(data, feature_column, a='stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 1', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_application"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

## Trench 2

In [ ]:
sq = """ 
WITH
  base AS (
    SELECT
      r.customer_id,
      r.digitalLoanAccountId,
      r.ln_appln_submit_datetime,
      r.trench_category,
      r.demo_score,
      r.apps_score,
      r.credo_score,
      r.stack_score,
      r.cic_score,
      lower(r.ln_os_type)  osType,
      date(
        IF(
          lmt.new_loan_type = 'Flex-up',
          lmt.startApplyDateTime,
          lmt.termsAndConditionsSubmitDateTime))
        application_date,
       case when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) < '2025-06-01' then 'Dev_Train'
           when lower(r.ln_os_type) like '%android%' and date(ln_appln_submit_datetime) >= '2025-06-01' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) >= '2024-10-01' and date(ln_appln_submit_datetime) <= '2025-02-28' then 'Dev_Train'
           when lower(r.ln_os_type) like '%ios%' and date(ln_appln_submit_datetime) > '2025-02-28' and date(ln_appln_submit_datetime) <= '2025-11-30' then 'Dev_Test' 
           end Data_selection
      -- CASE
      --   WHEN dev_split LIKE 'Dev_Train' THEN 'Dev_Train'
      --   WHEN dev_split LIKE 'Dev_Test' THEN 'Dev_Test'
      --   ELSE dev_split
      --   END Data_selection
    FROM
      `worktable_data_analysis.cash_alpha_trench2_applied_loans_backscored_20241001_20251130`
        r
    LEFT JOIN `risk_credit_mis.loan_master_table` lmt
      ON lmt.digitalLoanAccountId = r.digitalLoanAccountId
    WHERE stack_score is not null
  ),
  b1 as 
(SELECT 
customer_id,
digitalLoanAccountId,
ln_appln_submit_datetime,
trench_category,
demo_score,
apps_score,
credo_score,
stack_score,
cic_score,
osType,
application_date,
Data_selection,
FROM base
WHERE Data_selection IS NOT NULL
)
select * from b1;
"""

data = client.query(sq).to_dataframe(progress_bar_type='tqdm')
print(f"The shape of the dataframe is:\t {data.shape}")

In [ ]:
feature_column = ['demo_score', 'apps_score', 'credo_score', 'cic_score', 'stack_score']

dfd = transform_data_v1_1(data, feature_column, a='stack_score', modelDisplayName='Alpha-Cash-Stack-Model', tc='Trench 2', subscription_name = 'Cash December APPs Models Upgrade') 
print(f"the shape of the transformed dataframe is:\t {dfd.shape}")
dfd.info()

In [ ]:
dfd.sample(5)

In [ ]:
result = dfd.groupby('Data_selection').agg(
    digitalLoanAccountId_count=('digitalLoanAccountId', 'count'),
    Application_date_min=('Application_date', 'min'),
    Application_date_max=('Application_date', 'max')
).reset_index()

result

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_application"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dfd, table_id, job_config=job_config)
job.result() 

# 🪦💀 Graveyard

#### PSI Functions new

In [ ]:
# ## Updated on 27-10-2025 - Modified for Training Period Baseline
# import pandas as pd
# import numpy as np
# from typing import List, Dict, Tuple
# import warnings
# warnings.filterwarnings('ignore')

# def identify_feature_types(df: pd.DataFrame, feature_list: List[str]) -> Dict[str, List[str]]:
#     """
#     Identify categorical and numerical features from the feature list.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to classify

#     Returns:
#     --------
#     Dict with 'categorical' and 'numerical' keys containing respective feature lists
#     """
#     categorical_features = []
#     numerical_features = []

#     for feature in feature_list:
#         if feature not in df.columns:
#             print(f"Warning: Feature '{feature}' not found in dataframe")
#             continue

#         # Check if feature is numeric
#         if pd.api.types.is_numeric_dtype(df[feature]):
#             # If unique values are less than 15 and all integers, treat as categorical
#             unique_vals = df[feature].nunique()
#             if unique_vals < 15 and df[feature].dropna().apply(lambda x: x == int(x) if isinstance(x, (int, float)) else False).all():
#                 categorical_features.append(feature)
#             else:
#                 numerical_features.append(feature)
#         else:
#             categorical_features.append(feature)

#     return {
#         'categorical': categorical_features,
#         'numerical': numerical_features
#     }


# def create_bins_for_features(df: pd.DataFrame,
#                              numerical_features: List[str],
#                              categorical_features: List[str],
#                              train_period_df: pd.DataFrame) -> Dict:
#     """
#     Create bins for numerical features (deciles with fallback) and categorical features (top 6 + others)
#     based on the entire training period data.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Full input dataframe
#     numerical_features : List[str]
#         List of numerical features
#     categorical_features : List[str]
#         List of categorical features
#     train_period_df : pd.DataFrame
#         Training period dataframe (June 2024 to March 2025)

#     Returns:
#     --------
#     Dictionary containing binning information for each feature
#     """
#     binning_info = {}

#     # Create bins for numerical features with fallback strategy
#     for feature in numerical_features:
#         valid_data = train_period_df[feature].dropna()

#         if len(valid_data) == 0:
#             binning_info[feature] = {'type': 'numerical', 'bins': None, 'bin_ranges': {}}
#             continue

#         bins = None
#         bin_count = None

#         # Try 10 bins (deciles)
#         try:
#             test_bins = np.percentile(valid_data, np.arange(0, 101, 10))
#             test_bins = np.unique(test_bins)
#             if len(test_bins) >= 11:  # 11 edges = 10 bins
#                 bins = test_bins
#                 bin_count = 10
#         except Exception as e:
#             pass

#         # If 10 bins not possible, try 5 bins
#         if bins is None:
#             try:
#                 test_bins = np.percentile(valid_data, np.arange(0, 101, 20))
#                 test_bins = np.unique(test_bins)
#                 if len(test_bins) >= 6:  # 6 edges = 5 bins
#                     bins = test_bins
#                     bin_count = 5
#             except Exception as e:
#                 pass

#         # If 5 bins not possible, try 3 bins
#         if bins is None:
#             try:
#                 test_bins = np.percentile(valid_data, [0, 33.33, 66.67, 100])
#                 test_bins = np.unique(test_bins)
#                 if len(test_bins) >= 4:  # 4 edges = 3 bins
#                     bins = test_bins
#                     bin_count = 3
#             except Exception as e:
#                 pass

#         # If still no bins possible, use equal distance bins of 5
#         if bins is None:
#             print(f"Warning: Feature '{feature}' has insufficient variance - cannot create standard bins")
#             print(f"Feature '{feature}': Using equal distance bins of 5")

#             min_val = valid_data.min()
#             max_val = valid_data.max()

#             # Create 5 equal distance bins
#             bins = np.linspace(min_val, max_val, 6)  # 6 edges = 5 bins
#             bins = np.unique(bins)
#             bin_count = len(bins) - 1

#             # If all values are the same, add slight buffer
#             if bin_count == 1:
#                 bins = np.array([min_val - 0.1, min_val, min_val + 0.1])
#                 bin_count = 2
#                 print(f"Feature '{feature}': Constant value ({min_val}). Created 2 equal distance bins with buffer")

#         # Add infinity edges to capture all values
#         bins = bins.copy()
#         bins[0] = -np.inf
#         bins[-1] = np.inf

#         print(f"Feature '{feature}': Created {bin_count} bins")

#         # Create bin ranges dictionary
#         bin_ranges = {}
#         for i in range(len(bins)-1):
#             bin_name = f"Bin_{i+1}"
#             bin_ranges[bin_name] = {
#                 'min': bins[i],
#                 'max': bins[i+1],
#                 'range_str': f"[{bins[i]:.2f}, {bins[i+1]:.2f}]" if not np.isinf(bins[i]) and not np.isinf(bins[i+1]) else f"({bins[i]}, {bins[i+1]})"
#             }

#         binning_info[feature] = {
#             'type': 'numerical',
#             'bins': bins,
#             'bin_ranges': bin_ranges,
#             'bin_count': bin_count
#         }

#     # Create bins for categorical features (top 6 + others) using training period
#     for feature in categorical_features:
#         value_counts = train_period_df[feature].value_counts()
#         unique_categories = value_counts.index.tolist()
#         print(f"Unique categories: {unique_categories}")

#         if len(unique_categories) <= 6:
#             # Treat each category as a separate bin
#             top_categories = unique_categories
#         else:
#             # Use top 6 categories only
#             top_categories = value_counts.nlargest(6).index.tolist()

#         print(f"Top categories for feature '{feature}': {top_categories}")

#         binning_info[feature] = {
#                 'type': 'categorical',
#                 'top_categories': top_categories,
#                 'bin_ranges': {}  # No ranges for categorical
#             }

#     return binning_info


# def apply_binning(df: pd.DataFrame,
#                   feature: str,
#                   binning_info: Dict) -> pd.Series:
#     """
#     Apply binning to a feature based on binning information.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature : str
#         Feature name
#     binning_info : Dict
#         Binning information for the feature

#     Returns:
#     --------
#     pd.Series with binned values
#     """
#     if binning_info['type'] == 'numerical':
#         if binning_info['bins'] is None:
#             return pd.Series(['Missing'] * len(df), index=df.index)

#         bins = binning_info['bins']
#         labels = [f"Bin_{i+1}" for i in range(len(bins)-1)]

#         binned = pd.cut(df[feature],
#                        bins=bins,
#                        labels=labels,
#                        include_lowest=True,
#                        duplicates='drop')

#         # Handle nulls - convert to string and then replace
#         binned = binned.astype(str)
#         binned[df[feature].isna()] = 'Missing'

#         return binned

#     else:  # categorical
#         top_cats = binning_info['top_categories']

#         # Convert to string for consistent comparison
#         if pd.api.types.is_categorical_dtype(df[feature]):
#             feature_data = df[feature].astype(str)
#         else:
#             feature_data = df[feature].astype(str)

#         # Replace NaN string representation with 'Missing'
#         feature_data = feature_data.replace('nan', 'Missing')

#         # Convert top_cats to strings for comparison
#         top_cats_str = [str(cat) for cat in top_cats]

#         # Apply binning logic: use category name if in top_cats, else 'Others' (except for Missing)
#         binned = feature_data.apply(lambda x: x if x in top_cats_str else ('Others' if x != 'Missing' else 'Missing'))

#         return binned


# def calculate_psi(expected_pct: pd.Series,
#                   actual_pct: pd.Series,
#                   epsilon: float = 0.0001) -> float:
#     """
#     Calculate Population Stability Index with proper epsilon handling and renormalization.

#     Parameters:
#     -----------
#     expected_pct : pd.Series
#         Expected (baseline) percentages
#     actual_pct : pd.Series
#         Actual percentages
#     epsilon : float
#         Small value to avoid log(0)

#     Returns:
#     --------
#     PSI value
#     """
#     # Align indices
#     all_bins = expected_pct.index.union(actual_pct.index)
#     expected_pct = expected_pct.reindex(all_bins, fill_value=0)
#     actual_pct = actual_pct.reindex(all_bins, fill_value=0)

#     # Only add epsilon where values are zero
#     expected_pct = expected_pct.apply(lambda x: epsilon if x == 0 else x)
#     actual_pct = actual_pct.apply(lambda x: epsilon if x == 0 else x)

#     # Renormalize to ensure they sum to 1 after adding epsilon
#     expected_pct = expected_pct / expected_pct.sum()
#     actual_pct = actual_pct / actual_pct.sum()

#     # Calculate PSI
#     psi_value = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))

#     return psi_value


# def calculate_month_on_month_psi(df: pd.DataFrame,
#                                  feature_list: List[str],
#                                  segment_columns: List[str],
#                                  month_col: str = 'Application_month',
#                                  data_selection_col: str = 'Data_selection',
#                                  account_id_col: str = 'digitalLoanAccountId') -> pd.DataFrame:
#     """
#     Calculate PSI for each feature comparing training period (June 2024 to March 2025)
#     vs each month after March 2025, overall and by segments.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to calculate PSI for
#     segment_columns : List[str]
#         List of segment columns
#     month_col : str
#         Name of month column
#     data_selection_col : str
#         Name of data selection column (identifies train period)
#     account_id_col : str
#         Name of account ID column for counting distinct accounts

#     Returns:
#     --------
#     pd.DataFrame with PSI values with one row per feature-month-segment combination
#     """
#     # Create a copy to avoid modifying original
#     df = df.copy()

#     # Identify training and test periods
#     train_df = df[df[data_selection_col] == 'Train'].copy()
#     test_df = df[df[data_selection_col] != 'Train'].copy()

#     if len(train_df) == 0:
#         raise ValueError("No training data found. Check Data_selection column.")

#     print(f"Training period: {train_df[month_col].min()} to {train_df[month_col].max()}")
#     print(f"Test period: {test_df[month_col].min()} to {test_df[month_col].max()}")

#     # Identify feature types
#     feature_types = identify_feature_types(df, feature_list)

#     # Create binning strategy based on training period
#     binning_info = create_bins_for_features(
#         df,
#         feature_types['numerical'],
#         feature_types['categorical'],
#         train_df
#     )

#     # Get sorted test months
#     test_months = sorted(test_df[month_col].unique())

#     results = []

#     # Calculate overall PSI
#     for feature in feature_list:
#         if feature not in df.columns:
#             continue

#         # Apply binning to entire dataset
#         df[f'{feature}_binned'] = apply_binning(df, feature, binning_info[feature])
#         # print(f"Feature binned {df[f'{feature}_binned']}")
#         # Get training period distribution (baseline)
#         train_baseline = df[df[data_selection_col] == 'Train'][f'{feature}_binned'].value_counts(normalize=True)

#         # Calculate PSI for each test month
#         for month in test_months:
#             actual_dist = df[df[month_col] == month][f'{feature}_binned'].value_counts(normalize=True)
#             psi_value = calculate_psi(train_baseline, actual_dist)

#             # Calculate average percentages across all bins
#             expected_avg_pct = train_baseline.mean() * 100
#             actual_avg_pct = actual_dist.mean() * 100

#             # # Count distinct accounts for segment
#             # base_segment_count = train_segment[account_id_col].nunique()
#             # actual_segment_count = actual_segment[account_id_col].nunique()


#             results.append({
#                 'Feature': feature,
#                 'Feature_Type': binning_info[feature]['type'],
#                 'Segment_Column': 'Overall',
#                 'Segment_Value': 'All',
#                 'Month': f"{month}",
#                 'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                 'Current_Month': month,
#                 'Expected_Percentage': expected_avg_pct,
#                 'Actual_Percentage': actual_avg_pct,
#                 'PSI': psi_value
#             })

#     # Calculate PSI by segments
#     for segment_col in segment_columns:
#         if segment_col not in df.columns:
#             continue

#         segments = df[segment_col].dropna().unique()

#         for segment_val in segments:
#             segment_df = df[df[segment_col] == segment_val]

#             for feature in feature_list:
#                 if feature not in df.columns:
#                     continue

#                 # Get training period distribution for segment
#                 train_segment = segment_df[segment_df[data_selection_col] == 'Train']
#                 if len(train_segment) == 0:
#                     continue

#                 train_baseline = train_segment[f'{feature}_binned'].value_counts(normalize=True)

#                 # Calculate PSI for each test month
#                 for month in test_months:
#                     actual_segment = segment_df[segment_df[month_col] == month]
#                     if len(actual_segment) == 0:
#                         continue

#                     actual_dist = actual_segment[f'{feature}_binned'].value_counts(normalize=True)
#                     psi_value = calculate_psi(train_baseline, actual_dist)

#                     # Calculate average percentages across all bins
#                     expected_avg_pct = train_baseline.mean() * 100
#                     actual_avg_pct = actual_dist.mean() * 100

#                     # Count distinct accounts for segment
#                     base_segment_count = train_segment[account_id_col].nunique()
#                     actual_segment_count = actual_segment[account_id_col].nunique()

#                     results.append({
#                         'Feature': feature,
#                         'Feature_Type': binning_info[feature]['type'],
#                         'Segment_Column': segment_col,
#                         'Segment_Value': segment_val,
#                         'Month': f"{month}",
#                         'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                         'Current_Month': month,
#                         'Base_Count': base_segment_count,
#                         'Actual_Count': actual_segment_count,
#                         'Expected_Percentage': expected_avg_pct,
#                         'Actual_Percentage': actual_avg_pct,
#                         'PSI': psi_value
#                     })

#     return pd.DataFrame(results)


# def calculate_bin_level_psi(df: pd.DataFrame,
#                             feature_list: List[str],
#                             segment_columns: List[str],
#                             month_col: str = 'Application_month',
#                             data_selection_col: str = 'Data_selection',
#                             account_id_col: str = 'digitalLoanAccountId') -> pd.DataFrame:
#     """
#     Calculate bin-level PSI for each feature comparing training period
#     vs each month after March 2025, overall and by segments.

#     Parameters:
#     -----------
#     df : pd.DataFrame
#         Input dataframe
#     feature_list : List[str]
#         List of features to calculate PSI for
#     segment_columns : List[str]
#         List of segment columns
#     month_col : str
#         Name of month column
#     data_selection_col : str
#         Name of data selection column
#     account_id_col : str
#         Name of account ID column for counting distinct accounts

#     Returns:
#     --------
#     pd.DataFrame with bin-level PSI details including bin ranges
#     """
#     # Create a copy to avoid modifying original
#     df = df.copy()

#     # Identify training and test periods
#     train_df = df[df[data_selection_col] == 'Train'].copy()
#     test_df = df[df[data_selection_col] != 'Train'].copy()

#     if len(train_df) == 0:
#         raise ValueError("No training data found. Check Data_selection column.")

#     print(f"Training period: {train_df[month_col].min()} to {train_df[month_col].max()}")
#     print(f"Test period: {test_df[month_col].min()} to {test_df[month_col].max()}")

#     # Identify feature types
#     feature_types = identify_feature_types(df, feature_list)

#     # Create binning strategy based on training period
#     binning_info = create_bins_for_features(
#         df,
#         feature_types['numerical'],
#         feature_types['categorical'],
#         train_df
#     )

#     # Get sorted test months
#     test_months = sorted(test_df[month_col].unique())

#     results = []
#     epsilon = 0.0001

#     # Calculate overall bin-level PSI
#     for feature in feature_list:
#         if feature not in df.columns:
#             continue

#         # Apply binning to entire dataset
#         df[f'{feature}_binned'] = apply_binning(df, feature, binning_info[feature])
#         # print(df[f'{feature}_binned'])

#         # Get training period distribution (baseline)
#         train_baseline = df[df[data_selection_col] == 'Train'][f'{feature}_binned'].value_counts(normalize=True)

#         # Calculate bin-level PSI for each test month
#         for month in test_months:
#             month_data = df[df[month_col] == month]
#             actual_dist = month_data[f'{feature}_binned'].value_counts(normalize=True)

#             # Count distinct accounts
#             base_count = df[df[data_selection_col] == 'Train'][account_id_col].nunique()
#             actual_count = month_data[account_id_col].nunique()

#             # Get all bins
#             all_bins = train_baseline.index.union(actual_dist.index)

#             for bin_name in all_bins:
#                 # Simplified epsilon logic - no redundancy
#                 expected_pct = train_baseline.get(bin_name, 0)
#                 actual_pct = actual_dist.get(bin_name, 0)

#                 # Add epsilon only if zero
#                 expected_pct = epsilon if expected_pct == 0 else expected_pct
#                 actual_pct = epsilon if actual_pct == 0 else actual_pct

#                 # Calculate bin-level PSI
#                 bin_psi = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

#                 # Get bin range information
#                 bin_ranges = binning_info[feature]['bin_ranges']
#                 if bin_name in bin_ranges:
#                     bin_min = bin_ranges[bin_name]['min']
#                     bin_max = bin_ranges[bin_name]['max']
#                     bin_range = bin_ranges[bin_name]['range_str']
#                 else:
#                     # For categorical or special bins (Missing, Others)
#                     bin_min = None
#                     bin_max = None
#                     bin_range = bin_name

#                 results.append({
#                     'Feature': feature,
#                     'Feature_Type': binning_info[feature]['type'],
#                     'Segment_Column': 'Overall',
#                     'Segment_Value': 'All',
#                     'Month': f"{month}",
#                     'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                     'Current_Month': month,
#                     'Base_Count': base_count,
#                     'Actual_Count': actual_count,
#                     'Bin': bin_name,
#                     'Bin_Range': bin_range,
#                     'Bin_Min': bin_min,
#                     'Bin_Max': bin_max,
#                     'Base_Percentage': (train_baseline.get(bin_name, 0) * 100),
#                     'Actual_Percentage': (actual_dist.get(bin_name, 0) * 100),
#                     'Bin_PSI': bin_psi
#                 })

#     # Calculate bin-level PSI by segments
#     for segment_col in segment_columns:
#         if segment_col not in df.columns:
#             continue

#         segments = df[segment_col].dropna().unique()

#         for segment_val in segments:
#             segment_df = df[df[segment_col] == segment_val]

#             for feature in feature_list:
#                 if feature not in df.columns:
#                     continue

#                 # Get training period distribution for segment
#                 train_segment = segment_df[segment_df[data_selection_col] == 'Train']
#                 if len(train_segment) == 0:
#                     continue

#                 train_baseline = train_segment[f'{feature}_binned'].value_counts(normalize=True)

#                 # Calculate bin-level PSI for each test month
#                 for month in test_months:
#                     actual_segment = segment_df[segment_df[month_col] == month]
#                     if len(actual_segment) == 0:
#                         continue

#                     actual_dist = actual_segment[f'{feature}_binned'].value_counts(normalize=True)

#                     # Count distinct accounts for segment
#                     base_segment_count = train_segment[account_id_col].nunique()
#                     actual_segment_count = actual_segment[account_id_col].nunique()

#                     # Get all bins
#                     all_bins = train_baseline.index.union(actual_dist.index)

#                     for bin_name in all_bins:
#                         # Simplified epsilon logic - no redundancy
#                         expected_pct = train_baseline.get(bin_name, 0)
#                         actual_pct = actual_dist.get(bin_name, 0)

#                         # Add epsilon only if zero
#                         expected_pct = epsilon if expected_pct == 0 else expected_pct
#                         actual_pct = epsilon if actual_pct == 0 else actual_pct

#                         # Calculate bin-level PSI
#                         bin_psi = (actual_pct - expected_pct) * np.log(actual_pct / expected_pct)

#                         # Get bin range information
#                         bin_ranges = binning_info[feature]['bin_ranges']
#                         if bin_name in bin_ranges:
#                             bin_min = bin_ranges[bin_name]['min']
#                             bin_max = bin_ranges[bin_name]['max']
#                             bin_range = bin_ranges[bin_name]['range_str']
#                         else:
#                             # For categorical or special bins (Missing, Others)
#                             bin_min = None
#                             bin_max = None
#                             bin_range = bin_name

#                         results.append({
#                             'Feature': feature,
#                             'Feature_Type': binning_info[feature]['type'],
#                             'Segment_Column': segment_col,
#                             'Segment_Value': segment_val,
#                             'Month': f"{month}",
#                             'Base_Month': 'Train (Jun 2024 - Mar 2025)',
#                             'Current_Month': month,
#                             'Base_Count': base_segment_count,
#                             'Actual_Count': actual_segment_count,
#                             'Bin': bin_name,
#                             'Bin_Range': bin_range,
#                             'Bin_Min': bin_min,
#                             'Bin_Max': bin_max,
#                             'Base_Percentage': (train_baseline.get(bin_name, 0) * 100),
#                             'Actual_Percentage': (actual_dist.get(bin_name, 0) * 100),
#                             'Bin_PSI': bin_psi
#                         })

#     return pd.DataFrame(results)

# End